# The `bq` command line and scripting

`bq` is BigQuery’s Python-based command-line interface. It wraps the same REST API the
console and the client libraries use.

> ⚠️ **These `bq` commands are documented, not executed here.** `bq` and `gcloud` are not
> installed in the environment where this notebook was written. The syntax and flags come from
> the official
> [BigQuery CLI reference](https://docs.cloud.google.com/bigquery/docs/reference/bq-cli-reference).
> Run them yourself and compare the output.

Everything marked **Runs locally** below executes here, with no account.


## 0. Setup


In [ ]:
import sys
from pathlib import Path

# The course's pure core modules. No installation and no cloud SDK — that is
# the whole point of the split.
COURSE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))

from cost_model import (
    BIGQUERY_ONDEMAND_PER_TB,
    BIGQUERY_SLOT_HOUR,
    compare_architectures,
    estimate,
    format_comparison,
    monthly_runway,
)
from pipeline_logic import (
    aggregate_events,
    bucket_by_window,
    dedup_key,
    deduplicate,
    estimated_scan_bytes,
    partition_for_date,
    partition_for_event,
)

print("core modules on the path:", (COURSE_ROOT / "project" / "core").exists())


## 1. The command set

```text
bq mk       create a dataset, table, view, materialized view, connection, reservation
bq load     load data into a table
bq query    run a query
bq ls       list datasets, tables, jobs, models, routines
bq show     show details of a dataset, table, or job
bq rm       remove a dataset, table, or job
bq cp       copy a table
```

`bq mk` needs a **resource-type flag** to know what to create.


## 2. Everyday commands

```bash
# Set the project once, so every later command inherits it.
gcloud config set project YOUR_PROJECT_ID

# Create a dataset.
bq mk --dataset --location=YOUR_REGION YOUR_DATASET

# List what exists.
bq ls --datasets=true --format=pretty
bq ls --format=pretty YOUR_DATASET
bq show --format=pretty YOUR_DATASET.sales

# Run a query, CSV out.
bq query --use_legacy_sql=false --format=csv '
SELECT store_id, SUM(sales_amount) AS revenue
FROM `YOUR_PROJECT_ID.YOUR_DATASET.sales`
WHERE sale_date = "2026-11-13"
GROUP BY store_id
ORDER BY revenue DESC
'
```

**`--use_legacy_sql=false` should be in `.bigqueryrc`, not typed every time.**


## 3. `.bigqueryrc`

```ini
[query]
--use_legacy_sql=false
--max_rows=100
--maximum_bytes_billed=10000000

[load]
--skip_leading_rows=1
--autodetect=true
```

The default should be safe. Safety should not require remembering a flag.


## 4. The flags that matter

### `bq query`

| Flag | Purpose |
|---|---|
| `--use_legacy_sql=false` | Use GoogleSQL. Always pass this. |
| `--dry_run` | Validate and estimate bytes without executing. |
| `--maximum_bytes_billed=N` | Fail the query if it would exceed N bytes. |
| `--use_query_cache=false` | Bypass the result cache. |
| `--destination_table=...` | Write the result to a table. |
| `--parameter=...` | Bind a query parameter. |
| `--format=...` | `json`, `prettyjson`, `csv`, `pretty`, `none`, `text`. |

### `bq load`

| Flag | Purpose |
|---|---|
| `--source_format=FORMAT` | `CSV`, `NEWLINE_DELIMITED_JSON`, `AVRO`, `PARQUET`, `ORC` |
| `--schema={SCHEMA_FILE\|SCHEMA}` | Inline schema, or a file path |
| `--skip_leading_rows=N` | Skip the header |
| `--autodetect={true\|false}` | Infer the schema from the data |
| `--replace={true\|false}` | Erase existing data **and schema** first |
| `--field_delimiter=DELIMITER` | Or `-F=DELIMITER` |
| `--null_marker=STRING` | The string that means NULL |
| `--allow_jagged_rows={true\|false}` | Tolerate rows with trailing missing fields |
| `--max_bad_records=MAX` | Tolerate N bad rows before failing |


## 5. Dry run — a query you have never run before

```bash
bq query --use_legacy_sql=false --dry_run --format=pretty '
SELECT COUNT(*) FROM `YOUR_PROJECT_ID.YOUR_DATASET.sales`
'
```

This validates the query and returns the bytes it would scan, **without executing it**. A dry
run on a badly written query over a 40 TB table costs nothing and tells you it would have cost
$250.

**Runs locally** — the same habit, against the course’s cost model:


In [ ]:
# A dry run, expressed as a number the pipeline can act on.
DRY_RUN_TB = 40.0
dry_run_bytes = int(DRY_RUN_TB * 1024**4)
dry_run_cost = estimate("bigquery", scanned_tb=DRY_RUN_TB, apply_free_tier=False).total

print(f"dry run: {dry_run_bytes:,} bytes would be scanned")
print(f"dry run: ${dry_run_cost:.2f}")
print()
print("execute?", "NO - too expensive" if dry_run_cost > 250 else "yes")


## 6. The idempotent creation script

```bash
#!/usr/bin/env bash
set -euo pipefail

PROJECT="YOUR_PROJECT_ID"
DATASET="YOUR_DATASET"
LOCATION="YOUR_REGION"

if ! bq mk --dataset --location="${LOCATION}" "${PROJECT}:${DATASET}" 2>/dev/null; then
  echo "Dataset ${DATASET} already exists — continuing."
fi

bq rm --table --recursive=true "${PROJECT}:${DATASET}.sales" 2>/dev/null || true

bq mk --table \
  --location="${LOCATION}" \
  --schema=transaction_id:STRING,store_id:STRING,sale_date:DATE,customer_id:STRING,sales_amount:NUMERIC \
  --time_partitioning=sale_date \
  --clustering_fields=store_id \
  "${PROJECT}:${DATASET}.sales"
```

**Runs locally** — the same three properties, without BigQuery:


In [ ]:
# Idempotency, modelled: the same input always produces the same output.
batch = [
    {"event_id": "evt-0001", "event_dt": "2026-01-15", "amount_eur": 10.0},
    {"event_id": "evt-0002", "event_dt": "2026-01-15", "amount_eur": 20.0},
    {"event_id": "evt-0001", "event_dt": "2026-01-15", "amount_eur": 10.0},
]

def load_once(rows):
    """Stand-in for `bq load --replace=true`: a deterministic starting point."""
    return deduplicate(rows, partition="20260115")

first = load_once(batch)
second = load_once(batch)

print(f"run 1: {len(first)} rows")
print(f"run 2: {len(second)} rows")
print(f"identical: {first == second}")


## 7. Load, verify, assert

Never finish a load job with "the command exited 0". The pattern is: load, verify the
invariants, then fail loudly if they do not hold.

```bash
read -r ROWS DISTINCT <<<"$(bq query --use_legacy_sql=false --format=csv '
  SELECT COUNT(*), COUNT(DISTINCT transaction_id)
  FROM `YOUR_PROJECT_ID.YOUR_DATASET.sales`
  WHERE sale_date = "2026-11-13"
' | tail -n 1 | tr ',' ' ')"

if [ "${ROWS}" -ne "${DISTINCT}" ]; then
  echo "FAIL: duplicate transaction ids" >&2
  exit 1
fi
```

**Runs locally** — the assertion itself:


In [ ]:
def verify(rows, partition):
    """The invariant: distinct keys must equal rows once deduplicated.

    This is the check that catches a non-idempotent load or a double-delivered
    stream. If the loader is idempotent, the deduplicated set has no repeats.
    """
    unique = deduplicate(rows, partition=partition)
    rows_n = len(unique)
    distinct_n = len({dedup_key(r["event_id"], partition) for r in unique})
    return rows_n, distinct_n, rows_n == distinct_n

# A clean batch: one row per transaction.
clean = [
    {"event_id": "evt-0001", "event_dt": "2026-01-15", "amount_eur": 10.0},
    {"event_id": "evt-0002", "event_dt": "2026-01-15", "amount_eur": 20.0},
]
rows_n, distinct_n, ok = verify(clean, "20260115")
print(f"clean batch  -> rows={rows_n} distinct={distinct_n} ok={ok}")
assert ok, "duplicate transaction ids — failing the job"
print("assertion passed")


## 8. The Python client library, when to switch

Reach for Python rather than shelling out when the logic has loops, conditionals or retries;
when you need the Storage Write API with a write stream ID for exactly-once delivery; when you
want typed parameters instead of string interpolation; or when you want
`job.total_bytes_billed` programmatically, for a cost assertion.

```python
from google.cloud import bigquery

client = bigquery.Client()
job = client.query(
    """
    SELECT store_id, SUM(sales_amount) AS revenue
    FROM `retail.analytics.sales`
    WHERE sale_date = @sale_date
    GROUP BY store_id
    """,
    job_config=bigquery.QueryJobConfig(
        use_legacy_sql=False,
        maximum_bytes_billed=10_000_000_000,
        query_parameters=[
            bigquery.ScalarQueryParameter("sale_date", "DATE", "2026-11-13"),
        ],
    ),
)
rows = list(job.result())
print(f"{len(rows)} rows, {job.total_bytes_billed} bytes billed")
```

**Never build SQL by string-interpolating a user value.** Query parameters are correct by
construction.


## What you should be able to say now

1. The `bq` command set, and why `bq mk` needs a resource-type flag.
2. Why `--use_legacy_sql=false` belongs in `.bigqueryrc`.
3. What `--dry_run` gives you, and why it is free.
4. Why `maximumBytesBilled` belongs on every job.
5. The three properties that make a script production-grade: idempotent, bounded, verifying.
6. Why query parameters, not string interpolation.
